# Figure 3: feature embedding geometry

**Paper:** Section *Structural Properties of DIWECT Embeddings*, paragraph *Feature Embedding Geometry*, Figure `fig:exp:3-feature-geometry`.

| panel | output |
|---|---|
| (a) within-class squared distance over overall squared distance (no labels and no fitting are involved) | `figures/3-feature-geometry/figure3a-within-class.pdf` |
| (b) 2D metric MDS of the WECT and DIWECT distance matrices, coloured by shape and by pose bin | `figures/3-feature-geometry/figure3b-mds.pdf` |

**Data.** EAS2D generated on the fly: 60 random poses (angle uniform on $S^1$) of each of the five shapes per seed, 5 seeds (`default_rng(seed)`). Panel (a) is mean ± sd over the 5 seeds. Panel (b) embeds the last seed (seed 4) with MDS on the Euclidean distances of the raw features; points are coloured by true shape and by pose bin (5 equal bins of the rotation angle).

Runtime: about two minutes on a laptop CPU.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.manifold import MDS

import common as C

C.use_paper_style()
FOLDER = "3-feature-geometry"

SIZE, AREA = 64, 380
NUM_DIRS, NUM_HEIGHTS = 64, 48
N_PER_CLASS, N_SEEDS = 60, 5

METHODS = [("pixels", "baseline"), ("wect", "baseline"), ("wect_pca", "dim control"),
           ("diwect", "invariant"), ("wect_align", "oracle")]
ORDER = [m for m, _ in METHODS]
ROLE = dict(METHODS)
TICKS = {"pixels": "Pixels", "wect": "WECT", "wect_pca": "WECT\nPCA", "diwect": "DIWECT",
         "wect_align": "WECT\n+ align"}

shapes = C.make_shape_library(size=SIZE, area=AREA)
transform = C.WECTTransform(num_dirs=NUM_DIRS, num_heights=NUM_HEIGHTS)
K = len(shapes)

## Dataset and representations

`wect_pca` is the WECT projected by PCA to the DIWECT's width (48), a dimension control. `wect_align` is the WECT of each image rotated back by its true angle (an oracle).

In [2]:
def make_dataset(seed):
    rng = np.random.default_rng(seed)
    images, labels, angles = [], [], []
    for label, img in enumerate(shapes.values()):
        for _ in range(N_PER_CLASS):
            a = rng.uniform(0.0, 360.0)
            images.append(C.rotate_image(img, a))
            labels.append(label)
            angles.append(a)
    return images, np.asarray(labels), np.asarray(angles)


def build(images, angles):
    W = np.stack([transform.wect(im) for im in images])
    flat = W.reshape(len(W), -1)
    aligned = np.stack([transform.wect(C.rotate_image(im, -a)) for im, a in zip(images, angles)])
    return {
        "pixels": np.stack([im.ravel() for im in images]),
        "wect": flat,
        "wect_pca": PCA(n_components=NUM_HEIGHTS).fit_transform(flat),
        "diwect": W.mean(axis=1),
        "wect_align": aligned.reshape(len(W), -1),
    }


def within_group_ratio(d2, groups):
    """Mean within-group squared distance over the mean overall one (off-diagonal pairs)."""
    off = ~np.eye(len(d2), dtype=bool)
    same = (groups[:, None] == groups[None, :]) & off
    return float(d2[same].mean() / d2[off].mean())

## Within-class distances, over 5 seeds

In [3]:
rows = []
for seed in range(N_SEEDS):
    images, y, angles = make_dataset(seed)
    pose = np.minimum((angles / (360.0 / K)).astype(int), K - 1)   # K pose bins
    feats = build(images, angles)
    d2 = {name: cdist(feats[name], feats[name], "sqeuclidean") for name in ORDER}
    rows += [{"method": name, "seed": seed, "within_class": within_group_ratio(d2[name], y)} for name in ORDER]
    print(f"  seed {seed} done", flush=True)
last = {"d2": d2, "y": y, "pose": pose}   # the MDS panel uses the last seed

df = pd.DataFrame(rows)
agg = df.groupby("method").within_class.agg(["mean", "std"]).reindex(ORDER)
print(agg.round(3).to_string())

  seed 0 done


  seed 1 done


  seed 2 done


  seed 3 done


  seed 4 done


             mean    std
method                  
pixels      0.919  0.001
wect        0.888  0.004
wect_pca    0.847  0.004
diwect      0.200  0.047
wect_align  0.244  0.007


## MDS embeddings of the last seed

Metric MDS on the distance matrices themselves (not PCA on the features), so the picture is faithful to the metric.

In [4]:
embeddings = {name: MDS(n_components=2, metric="precomputed", init="random", random_state=0,
                        normalized_stress="auto").fit_transform(np.sqrt(last["d2"][name]))
              for name in ("wect", "diwect")}

## The figure panels

In [5]:
# (a) within-class d^2 / overall d^2
fig, ax = plt.subplots(figsize=(5.0, 3.8))
xs = np.arange(len(ORDER))
bars = ax.bar(xs, agg["mean"], 0.62, yerr=agg["std"], capsize=2, color=[C.color(ROLE[m]) for m in ORDER])
C.bar_labels(ax, bars)
ax.axhline(1.0, ls="--", lw=1, color="k")
ax.annotate("class explains nothing", (len(ORDER) - 0.5, 1.02), ha="right", fontsize=7.5)
ax.set_xticks(xs); ax.set_xticklabels([TICKS[m] for m in ORDER])
ax.set_ylim(0, 1.25)
ax.set_ylabel("within-class $d^2$ / overall $d^2$")
C.save_pdf(fig, FOLDER, "figure3a-within-class")


# (b) MDS embeddings, 2 x 2: WECT on top, DIWECT below
MARKERS = ["o", "s", "^", "D", "P"]  # one per shape class, so classes differ by colour and shape


def embed(ax, Z, colors, title, cmap, legend=None, markers=None):
    if markers is None:
        ax.scatter(Z[:, 0], Z[:, 1], c=colors, s=11, cmap=cmap, alpha=0.85, edgecolors="none")
    else:
        norm = plt.Normalize(colors.min(), colors.max())
        for k in np.unique(colors):
            sel = colors == k
            ax.scatter(Z[sel, 0], Z[sel, 1], color=plt.get_cmap(cmap)(norm(k)), s=13,
                       marker=markers[int(k) % len(markers)], alpha=0.85, edgecolors="none")
    lo, hi = np.percentile(Z, [1, 99], axis=0)
    pad = 0.12 * (hi - lo + 1e-9)
    ax.set_xlim(lo[0] - pad[0], hi[0] + pad[0]); ax.set_ylim(lo[1] - pad[1], hi[1] + pad[1])
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=9.5)
    if legend:
        handles = [plt.Line2D([], [], marker=MARKERS[i % len(MARKERS)] if markers else "o", ls="", ms=5,
                              color=plt.get_cmap(cmap)(i / max(len(legend) - 1, 1)))
                   for i in range(len(legend))]
        ax.legend(handles, legend, fontsize=6.5, ncol=2, loc="best")


fig, axes = plt.subplots(2, 2, figsize=(7.0, 6.6))
for row, rep in enumerate(("wect", "diwect")):
    embed(axes[row, 0], embeddings[rep], last["y"], f"{C.label(rep)}: shape", "tab10", legend=list(shapes), markers=MARKERS)
    embed(axes[row, 1], embeddings[rep], last["pose"], f"{C.label(rep)}: pose bin", "twilight")
C.save_pdf(fig, FOLDER, "figure3b-mds")

saved -> figures/3-feature-geometry/figure3a-within-class.pdf
saved -> figures/3-feature-geometry/figure3b-mds.pdf
